In [3]:
# Imports
import cv2 as cv
import pandas as pd

In [4]:
# Load Video
INPUT_VIDEO = "../data/input/videoplayback.mp4"

In [7]:
# Open video file
cap = cv.VideoCapture(INPUT_VIDEO)

print("Video loaded successfully!")

if not cap.isOpened():
    print("Error: Cannot open video file")
    exit()

Video loaded successfully!


In [25]:
# Get videos properties
fps = cap.get(cv.CAP_PROP_FPS)
width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv.CAP_PROP_FRAME_COUNT))

print(f"FPS: {fps}, Size: {width}x{height}, Frames: {frame_count}")

FPS: 29.97002997002997, Size: 480x360, Frames: 1427


In [26]:
NUMBER_OF_FRAMES_TO_EXTR = 4
frame_indices = [int(frame_count * i / NUMBER_OF_FRAMES_TO_EXTR) for i in range(NUMBER_OF_FRAMES_TO_EXTR)]
print(frame_indices)

[0, 356, 713, 1070]


In [27]:
frames = []

for frame_id in frame_indices:
    cap.set(cv.CAP_PROP_POS_FRAMES, frame_id)
    ret, frame = cap.read()

    if ret:
        frames.append(cv.cvtColor(frame, cv.COLOR_BGR2RGB))

print(frames)
cap.release()


[array([[[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       ...,

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]], shape=(360, 480, 3), dtype=uint8), array([[[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0

# Segmentacja

In [8]:
# Get videos properties
fps = cap.get(cv.CAP_PROP_FPS)
width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv.CAP_PROP_FRAME_COUNT))

print(f"FPS: {fps}, Size: {width}x{height}, Frames: {frame_count}")

FPS: 29.97002997002997, Size: 480x360, Frames: 1427


In [9]:
chunk_time = 10
chunk_frames = 4
stride = 2

chunk_total_frames = int(chunk_time * fps)
print(f"{chunk_total_frames}fps")
overlap_frames = int(stride * fps)


299fps


In [10]:
# Krok (stride) - o ile klatek w przód przesuwa się "okno" z każdym nowym chunkiem
stride_frames = chunk_total_frames - overlap_frames
if stride_frames <= 0:
    raise ValueError("Stride time has to be shorter than chunk time.")

In [11]:
frame_stamp = chunk_total_frames / chunk_frames
print(f"{frame_stamp}fps")

74.75fps


In [12]:
all_chunks_indices = []

# Z użyciem parametru "step" (stride_frames)
for start_frame in range(0, frame_count, stride_frames):
    
    # Wyznaczamy koniec chunku (nie przekraczając końca wideo)
    end_frame = min(start_frame + chunk_total_frames, frame_count)
    current_chunk_length = end_frame - start_frame
    
    # Modyfikacja: używamy (chunk_frames - 1), aby ostatnia próbka 
    # była zawsze na samym końcu chunku/wideo.
    # Wymaga to max(1, ...), aby nie podzielić przez zero gdybyśmy chcieli tylko 1 klatkę.
    current_chunk_indices = [
        int(start_frame + ((current_chunk_length - 1) * i / max(1, chunk_frames - 1))) 
        for i in range(chunk_frames)
    ]
    
    # Zabezpieczenie przed duplikatami
    current_chunk_indices = sorted(list(set(current_chunk_indices)))
    all_chunks_indices.append(current_chunk_indices)
    
    # Jeśli dotarliśmy do końca nagrania - kończymy
    if end_frame == frame_count:
        break


In [54]:
print(f"Długość chunku: {chunk_total_frames}, Overlap: {overlap_frames}, Krok: {stride_frames}\n")
for idx, chunk in enumerate(all_chunks_indices):
    print(f"Chunk {idx + 1}: {chunk}")

Długość chunku: 299, Overlap: 59, Krok: 240

Chunk 1: [0, 99, 198, 298]
Chunk 2: [240, 339, 438, 538]
Chunk 3: [480, 579, 678, 778]
Chunk 4: [720, 819, 918, 1018]
Chunk 5: [960, 1059, 1158, 1258]
Chunk 6: [1200, 1275, 1350, 1426]
